In [1]:
import torch
import ttnn

2026-02-09 11:32:13.322 | DEBUG    | ttnn:<module>:77 - Initial ttnn.CONFIG:
Config{cache_path=/root/.cache/ttnn,model_cache_path=/root/.cache/ttnn/models,tmp_dir=/tmp/ttnn,enable_model_cache=false,enable_fast_runtime_mode=true,throw_exception_on_fallback=false,enable_logging=false,enable_graph_report=false,enable_detailed_buffer_report=false,enable_detailed_tensor_report=false,enable_comparison_mode=false,comparison_mode_should_raise_exception=false,comparison_mode_pcc=0.9999,root_report_path=generated/ttnn/reports,report_name=std::nullopt,std::nullopt}


In [6]:
k_prefill = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill_1_new_token/only_prefill_k_0.tensorbin")
q_prefill = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill_1_new_token/only_prefill_q_0.tensorbin")
v_prefill = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill_1_new_token/only_prefill_v_0.tensorbin")

In [7]:
k_prefill.shape

torch.Size([1, 32, 17, 64])

In [8]:
k_decode = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill_1_new_token/decode_k_16.tensorbin")
q_decode= torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill_1_new_token/decode_q_16.tensorbin")
v_decode = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill_1_new_token/decode_v_16.tensorbin")

In [9]:
k_decode.shape

torch.Size([1, 1, 32, 64])

In [23]:
k_only_prefill_og = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill/only_prefill_k_0.tensorbin")
q_only_prefill_og = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill/only_prefill_q_0.tensorbin")
v_only_prefill_og = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill/only_prefill_v_0.tensorbin")

In [50]:
k_decode = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill_1_new_token/decode_k_final_16.tensorbin")
q_decode= torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill_1_new_token/decode_q_final_16.tensorbin")
v_decode = torch.load("/tt-metal/models/experimental/indusproject/after_l1/prefill_1_new_token/decode_v_final_16.tensorbin")

In [51]:
k_only_prefill_og.shape

torch.Size([1, 32, 25, 64])

In [54]:
k_only_prefill_og[:, :, 16, :]

tensor([[[ 0.8516,  1.3984, -0.9375,  ...,  0.6992,  0.0425,  0.2256],
         [ 1.7734,  1.4766,  2.4844,  ..., -0.1846,  0.4785,  0.1377],
         [ 0.3945, -0.0933, -0.9375,  ..., -0.5547, -0.0374, -0.2988],
         ...,
         [-1.1406,  0.6719,  0.6953,  ...,  1.3281,  0.0454,  0.1001],
         [ 0.0679, -0.3242,  0.2598,  ..., -1.3047, -0.5625, -1.5781],
         [ 1.1719,  0.3516,  1.5078,  ...,  0.3184,  0.8164, -0.1367]]],
       dtype=torch.bfloat16)

In [55]:
k_decode[0]

tensor([[[ 0.8438,  1.3984, -0.9180,  ...,  0.7344,  0.0508,  0.1934],
         [ 1.7812,  1.4844,  2.5000,  ..., -0.1709,  0.4863,  0.1216],
         [ 0.4102, -0.0913, -0.9492,  ..., -0.5703, -0.0354, -0.3145],
         ...,
         [-1.1328,  0.6758,  0.6836,  ...,  1.3438,  0.0425,  0.1367],
         [ 0.0674, -0.3379,  0.2734,  ..., -1.3125, -0.6172, -1.6016],
         [ 1.1250,  0.3262,  1.4844,  ...,  0.3164,  0.8047, -0.1279]]],
       dtype=torch.bfloat16)

In [56]:
k = k_only_prefill_og[:, :, 16, :]- k_decode[0]

In [71]:
k[0][0]

tensor([ 0.0078,  0.0000, -0.0195, -0.0068,  0.0312,  0.0156, -0.0332,  0.0039,
         0.0039, -0.0117, -0.0156,  0.0000, -0.0273,  0.0000,  0.0273, -0.0020,
        -0.0107, -0.0195, -0.0186, -0.0273,  0.0000, -0.0078,  0.0123, -0.0117,
        -0.0312, -0.0469,  0.0039, -0.0391, -0.0039,  0.0312,  0.0156, -0.0151,
         0.0312, -0.0312, -0.0117, -0.0625, -0.0098, -0.0625,  0.0039,  0.0371,
        -0.0156,  0.0312,  0.0039, -0.0078, -0.0625,  0.0195, -0.0156,  0.0063,
        -0.0312, -0.0176,  0.0234,  0.0078, -0.0078,  0.0234, -0.0078,  0.0000,
         0.0078, -0.0156, -0.0234,  0.0000, -0.0039, -0.0352, -0.0083,  0.0322],
       dtype=torch.bfloat16)

In [59]:
prefill_embed = torch.load("/tt-metal/models/experimental/indusproject/after_l1/embed/prefill_embed_0.tensorbin")
prefill_embed.shape

torch.Size([1, 1, 17, 6144])

In [60]:
decode_embed = torch.load("/tt-metal/models/experimental/indusproject/after_l1/embed/decode_embed_16.tensorbin")
print(decode_embed.shape)

torch.Size([1, 1, 32, 6144])


In [61]:
prefill_embed[0, 0, -1, :]

tensor([-0.6406, -0.2773, -0.1953,  ...,  0.0057,  0.0034, -0.0505],
       dtype=torch.bfloat16)

In [62]:
decode_embed[0, 0, 0, :]

tensor([-0.6406, -0.2793, -0.1953,  ...,  0.0058,  0.0034, -0.0508],
       dtype=torch.bfloat16)

In [63]:
emb = prefill_embed[0, 0, -1, :] - decode_embed[0, 0, 0, :]

In [64]:
emb

tensor([ 0.0000e+00,  1.9531e-03,  0.0000e+00,  ..., -1.2207e-04,
         3.0518e-05,  2.4414e-04], dtype=torch.bfloat16)

In [65]:
decode_embed_reshaped = torch.load("/tt-metal/models/experimental/indusproject/after_l1/embed/decode_embed_reshaped_16.tensorbin")
print(decode_embed_reshaped.shape)

torch.Size([1, 1, 1, 6144])


In [67]:
emb_reshaped = prefill_embed[0, 0, -1, :] - decode_embed_reshaped[0, 0, 0, :]

In [68]:
emb_reshaped

tensor([ 0.0000e+00,  1.9531e-03,  0.0000e+00,  ..., -1.2207e-04,
         3.0518e-05,  2.4414e-04], dtype=torch.bfloat16)

In [ ]:
# gives error in paged_update_cahe: TT_FATAL: Expect batch in input tensor match the batch in cache tensor (assert.hpp:103). (without reshape)
# xqkv_fused.shape
# Shape([1, 1, 32, 6144])
# k.shape
# Shape([1, 32, 32, 64])
# keys.shape
# Shape([1, 32, 1024, 64])

In [ ]:

# (with reshape) no error
# xqkv_fused.shape
# Shape([1, 1, 32, 6144])

#after reshape
# xqkv_fused.shape
# Shape([1, 1, 1, 6144])
# xqkv_fused.padded_shape
# Shape([1, 1, 32, 6144])

# k.shape
# Shape([1, 1, 32, 64])
# keys.shape
# Shape([1, 32, 1024, 64])